# Что видит машина

По городу едет автомобиль. На нём пять камер по кругу и лазерный дальномер,
который вращается и меряет расстояние во все стороны. Запись длится двадцать секунд.

Мы не будем обучать модель. Все модели готовы и этих кадров раньше не видели.
Настраивать будем вопрос, который модели задаём.

Ячейки выполняются сверху вниз: кнопка слева от ячейки либо Shift и Enter вместе.
Настройки вводятся по ходу занятия, каждая под свою задачу.

## Что за данные

Waymo Open Perception — открытый набор для исследований автономного вождения,
выложенный компанией Waymo. Целиком это 2030 записей по двадцать секунд, снятых
в разных городах и погодных условиях. Мы работаем с одной записью.

| Прибор | Что даёт | Для чего обычно нужен |
| --- | --- | --- |
| Пять камер по кругу, десять кадров в секунду | цвет, форму, надписи. Расстояний не знает | обнаружение объектов, сегментация, чтение текста |
| Лазерный дальномер, трёхмерный | 144 878 точек за оборот, дальность от 4 до 74 м. Ни цвета, ни смысла | измерение расстояний и размеров, свободное пространство |

Дальномер вращается и меряет время возврата луча. Освещение ему безразлично,
ночью он работает так же, как днём, но человека от столба не отличает.
Камера различает смысл, но не расстояние. Поэтому их складывают.

Разметку делали люди, и она служит эталоном: рамки с номерами объектов на снимках,
класс каждой точки изображения, объёмные рамки вокруг машины.

## 1. Что в записи

Пять камер, у каждой 199 кадров. Разметку на них проставляли люди, и она нам известна.

In [ ]:
# правки во вспомогательном коде подхватываются без перезапуска ядра
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# ноутбук лежит в отдельной папке, а вспомогательный код в корне проекта
repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "stand").is_dir())
sys.path.insert(0, str(repo_root))

import matplotlib.pyplot as plt
from stand import data, run, viz

plt.rcParams["figure.dpi"] = 110
data.segment_summary()

## 2. Берём отрезок

Из двадцати секунд берём кусок и одну камеру.

| Настройка | Что задаёт |
| --- | --- |
| N_FRAMES | сколько кадров взять. Сорок кадров это четыре секунды |
| CAMERA | 1 передняя, 2 и 3 передние боковые, 4 и 5 боковые |
| START | с какого кадра записи начать |

In [ ]:
N_FRAMES = 40   # сколько кадров взять: от 1 до 199. Десять кадров это одна секунда
CAMERA   = 2    # какая камера: 1 передняя, 2 и 3 передние боковые, 4 и 5 боковые
START    = 0    # с какого кадра записи начать: от 0 до 198, но START + N_FRAMES не больше 199

clip = data.load_clip(n_frames=N_FRAMES, camera=CAMERA, start=START)

print(f"кадров: {len(clip)}, размер: {clip.size[0]}x{clip.size[1]}")
print("\nСколько объектов реально прошло за отрезок:")
print(clip.truth_counts().to_string())

viz.show(clip.frames[0], "Первый кадр отрезка");

## 3. Что видит каждая камера

Пять камер закрывают круг. Ниже их кадры, склеенные в одну ленту слева направо.
Стыки не сглаживаются: камеры смотрят под разными углами, склейка условная.

Загружается тот же отрезок, что задан настройками ниже, поэтому все ролики занятия
показывают одно и то же время. Ячейка читает пять записей подряд и выполняется дольше
остальных.

In [ ]:
from stand import multicam

cams = multicam.load_all(n_frames=N_FRAMES, start=START)
multicam.show_panorama(cams, frame=0);

## 4. Как это выглядит в движении

Тот же отрезок как обычное видео. Пока без моделей и без разметки.

In [ ]:
from stand import video

video.raw(clip)

## 5. Ищем словами

Модель принимает текстовый запрос. Никакого списка классов у неё нет: что напишете,
то и будет искать.

| Настройка | Что задаёт |
| --- | --- |
| QUERY | слово или фраза, которую ищем |
| THRESHOLD | граница уверенности. Объект с меньшей оценкой отбрасывается |
| MIN_TRACK_LEN | объект, замеченный в меньшем числе кадров, не засчитывается |

Запрос пишется по-английски: модель обучена на английском тексте и русских слов
не понимает. Проверено на этой записи, person находит объекты, человек не находит
ничего. Ошибки при этом не будет, просто пустой ответ.

| Что ищем | Что писать |
| --- | --- |
| человек, пешеход | person, pedestrian |
| легковая машина | car |
| любой транспорт | vehicle |
| грузовик | truck |
| автобус | bus |
| прочий крупный транспорт | other vehicle |
| прицеп | trailer |
| велосипед | bicycle |
| велосипедист | cyclist |
| мотоцикл | motorcycle |
| мотоциклист | motorcyclist |
| вещь при пешеходе | pedestrian object |
| дорожный знак | sign |
| светофор | traffic light |
| столб | pole |
| дорожный конус | construction cone |
| здание | building |
| дорога | road |
| разметка полосы | lane marker |
| надпись на асфальте | road marker |
| тротуар | sidewalk |
| растительность, дерево | vegetation, tree |
| небо | sky |
| земля, грунт | ground |
| птица | bird |
| животное | animal |

Перечень взят из разметки этой записи: люди размечали ровно эти классы, значит
объекты таких типов в сцене есть. Модель при этом классами не ограничена и найдёт
всё, что вы опишете словами: crane, fence, window, wheel, license plate, umbrella.

Работают и словосочетания: white car, person crossing the road, parked truck.

Расчёт занимает около двадцати секунд.

In [ ]:
QUERY         = "person"  # что ищем, по-английски. Список слов в таблице выше
THRESHOLD     = 0.5       # граница уверенности: от 0.0 до 1.0. Ниже границы объект отброшен
MIN_TRACK_LEN = 1         # сколько кадров объект должен продержаться: от 1 до N_FRAMES

result = run.find(clip, QUERY, conf=THRESHOLD, min_track_len=MIN_TRACK_LEN)

frame = 0
det = result.detections[result.detections["кадр"] == frame]
viz.show(viz.draw(clip.frames[frame], det, result.masks.get(frame)),
         f"Запрос {QUERY}, порог {THRESHOLD}, найдено {len(det)}");

## 6. Не теряет ли модель объекты

Тот же отрезок, но теперь с найденными объектами и их номерами.

Смотрите на номер и цвет. Пока объект сохраняет номер, модель считает его одним и тем же.
Если номер сменился, объект потерян и заведён заново. При подсчёте он будет посчитан дважды.

Ошибка эта встречается постоянно. В одном производственном проекте на девять реально
существующих линий разметки система выдала четырнадцать номеров.

In [ ]:
video.tracking(clip, result)

## 7. Как эту сцену разметил человек

Двадцать девять классов, проставленных вручную по точкам изображения: дорога, тротуар,
здания, растительность, столбы, знаки.

Такая разметка нужна по двум причинам. По ней обучают модели сегментации. И по ней
измеряют качество: имея эталон, можно посчитать, насколько точно модель обнаружила,
выделила и отнесла к классу каждый объект.

Разметка положена через кадр, поэтому ячейка сама выбирает подходящий.

In [ ]:
from stand import scene

labeled = scene.labeled_frames(clip)
scene.show_panoptic(clip, labeled[0])
scene.class_areas(clip, labeled[0]).head(10)

## 8. Поиск по всему кругу

Тот же запрос применяется ко всем пяти камерам сразу.

Ячейка считает по пяти записям подряд, поэтому выполняется дольше остальных.

In [ ]:
per_camera = multicam.search_all(cams, QUERY, conf=THRESHOLD)
multicam.show_panorama(cams, frame=0, results=per_camera);

Те же пять камер, но роликами. Видно, как объект уходит из одной камеры и появляется
в соседней, и что номер при этом переходе не сохраняется: каждая камера считает отдельно.

In [ ]:
video.cameras(cams, per_camera)

## 9. Лазерный дальномер

Дальше работает второй прибор. Он вращается и меряет расстояние до каждой точки
поверхности вокруг автомобиля. Что перед ним находится, прибор не понимает.

Сначала вид сверху: как выглядит сцена в плане. Красный треугольник это наш автомобиль,
дуги это следы лучей на дороге.

In [ ]:
from stand import lidar

lidar.bev(clip, frame=0);

## 10. Измерение и предсказание

Расстояние можно не только измерить прибором, но и предсказать по одной обычной
фотографии. Такая модель тоже существует и тоже берётся готовой.

Ниже три изображения рядом. Слева обычный снимок. В центре расстояние, предсказанное
моделью по этому снимку. Справа расстояние, измеренное дальномером.

Модель выдаёт относительную величину, а не метры. В метры она переводится по показаниям
дальномера. Отсюда следствие: камера становится измерительным прибором только после
привязки к эталону и точнее эталона быть не может.

In [ ]:
from stand import depth

fig, quality = depth.compare(clip, frame=0)
# quality

## 11. Объекты на плоскости во времени

Последний ролик сводит два прибора вместе.

Объекты найдены камерой по вашему запросу. По облаку точек поиск не ведётся: дальномер
не отличает человека от столба. Он отвечает на другой вопрос, сколько метров до объекта.
Связка идёт по контуру: берём измерения, попавшие внутрь выделенной области.

Серые точки это облако дальномера, оно круговое. Тонкие рамки это разметка людьми,
она тоже круговая и покрывает всё вокруг машины. Цветные метки с номерами это объекты,
найденные моделью, и только по выбранной камере. Линия за меткой это пройденный путь.

Расхождение между рамкой и меткой видно глазом, и оно закономерно. Проверено на этой
записи: для пешехода метка отстоит от рамки на 0,7 до 1,8 м при росте около метра,
для машины длиной 9,5 м на 3,5 до 4,5 м. Расхождение составляет примерно половину
длины объекта, и причина в физике измерения. Дальномер видит только обращённую к нему
поверхность, поэтому середина видимых точек смещена от середины объекта вперёд.
Прибор меряет то, что видит, а не то, что есть.

In [ ]:
video.lidar_tracks(clip, result)

## Что попробовать

Вернитесь к настройкам и поменяйте их. После смены выполните ячейки заново,
начиная с той, где настройка задана.

| Что изменить | Что посмотреть |
| --- | --- |
| QUERY на car или vehicle | сколько машин найдётся на той же записи |
| QUERY на traffic cone или pole | видит ли модель мелкие предметы |
| THRESHOLD на 0.3 и на 0.8 | какие объекты появляются и исчезают |
| CAMERA на 1, 4 или 5 | что видно с других сторон автомобиля |
| START на 100 | другой участок дороги и другие объекты |
| N_FRAMES на 80 | дольше ли держатся номера объектов |